# GP Travel Advisor - Retrain Staging

Notebook này dùng Colab để train model, lưu kết quả vào Google Drive và upload artifact lên Cloudflare R2.

Flow mặc định:

1. Mount Google Drive.
2. Cài dependency.
3. Clone repo.
4. Chạy `colab_retrain_pipeline.py --force` để train và upload R2.
5. Xem snapshot, RMSE, artifact, `tourist_user_map.csv`.
6. Restart `ai-service` để tải artifact mới từ R2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/GP-Retrain'
print('Drive root:', DRIVE_ROOT)

In [ ]:
!mkdir -p "$DRIVE_ROOT/state" "$DRIVE_ROOT/input" "$DRIVE_ROOT/output/data" "$DRIVE_ROOT/output/recommender_artifacts" "$DRIVE_ROOT/logs"

## Cài dependencies

Colab runtime mới cần cài lại package. Cell này có thể mất vài phút, nhất là `scikit-surprise`.

In [ ]:
!pip install -q supabase boto3 sentence-transformers scikit-surprise

## Lấy code

Nếu repo private, thay `YOUR_REPO_URL` bằng URL repo có token, hoặc upload thủ công folder `ai-service/retrain` vào Colab runtime.

In [ ]:
REPO_URL = 'YOUR_REPO_URL'  # ví dụ: https://github.com/<user>/<repo>.git
REPO_DIR = '/content/GP-Travel-Advisor-Backend'
RETRAIN_DIR = f'{REPO_DIR}/ai-service/retrain'

import os

if not os.path.exists(REPO_DIR):
    if REPO_URL == 'YOUR_REPO_URL':
        raise SystemExit(
            'Chưa có repo trong Colab. Hãy set REPO_URL rồi chạy lại cell này, '
            'hoặc upload repo/folder ai-service/retrain vào /content/GP-Travel-Advisor-Backend.'
        )
    !git clone "$REPO_URL" "$REPO_DIR"

if not os.path.exists(RETRAIN_DIR):
    raise SystemExit(f'Không tìm thấy {RETRAIN_DIR}. Kiểm tra repo clone đúng chưa.')

%cd $RETRAIN_DIR

## Cấu hình secrets

Không ghi secret thật vào notebook nếu notebook có thể được share. Cách gọn nhất là dùng Colab Secrets, rồi đọc qua `userdata.get(...)`.

Bạn cần các secret:

- `SUPABASE_URL`
- `SUPABASE_KEY`
- `R2_ENDPOINT_URL`
- `R2_ACCESS_KEY_ID`
- `R2_SECRET_ACCESS_KEY`
- `R2_BUCKET_NAME`

In [ ]:
import os
from google.colab import userdata

for key in [
    'SUPABASE_URL',
    'SUPABASE_KEY',
    'R2_ENDPOINT_URL',
    'R2_ACCESS_KEY_ID',
    'R2_SECRET_ACCESS_KEY',
    'R2_BUCKET_NAME',
]:
    value = userdata.get(key)
    if value:
        os.environ[key] = value

os.environ['COLAB_RETRAIN_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['RETRAIN_BASE_RATING_MATRIX_DIR'] = '/content/drive/MyDrive/Recommender System'

print('Configured keys:')
for key in ['SUPABASE_URL', 'SUPABASE_KEY', 'R2_ENDPOINT_URL', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY', 'R2_BUCKET_NAME']:
    print(key, 'OK' if os.environ.get(key) else 'MISSING')

## Train và upload R2

Cell này là bước chính: train model, lưu kết quả vào Drive và upload artifact lên Cloudflare R2 nếu R2 env đã cấu hình.

In [ ]:
!python colab_retrain_pipeline.py --force

## Xem kết quả train

Cell này đọc `snapshot.json`, `serve_manifest.json`, danh sách artifact và map UUID -> numeric id.

In [ ]:
import json
from pathlib import Path
import pandas as pd

root = Path(DRIVE_ROOT)
snapshot_path = root / 'output/data/snapshot.json'
manifest_path = root / 'output/recommender_artifacts/serve_manifest.json'
map_path = root / 'state/tourist_user_map.csv'

if snapshot_path.exists():
    snapshot = json.loads(snapshot_path.read_text(encoding='utf-8'))
    print('=== DATA SNAPSHOT ===')
    print(json.dumps(snapshot, ensure_ascii=False, indent=2))
else:
    print('Missing snapshot:', snapshot_path)

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('\n=== TRAIN METRICS ===')
    print(json.dumps(manifest.get('metrics', {}), ensure_ascii=False, indent=2))
    print('\n=== MODEL SHAPE ===')
    print('num_users =', manifest.get('num_users'))
    print('num_items =', manifest.get('num_items'))
    print('n_factors =', manifest.get('n_factors'))
    print('trained_at =', manifest.get('trained_at'))
    print('best_svd_params =', manifest.get('best_svd_params'))
else:
    print('Missing manifest:', manifest_path)

print('\n=== ARTIFACT FILES ===')
artifact_dir = root / 'output/recommender_artifacts'
for p in sorted(artifact_dir.glob('*')):
    print(p.name, f'{p.stat().st_size / 1024 / 1024:.2f} MB')

if map_path.exists():
    df_map = pd.read_csv(map_path)
    print('\n=== TOURIST USER MAP ===')
    print('rows =', len(df_map))
    display(df_map.head(10))
else:
    print('\nMissing tourist map:', map_path)

## Kiểm tra nhanh output data

Dùng cell này để nhìn thử Places và rating users/items.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path(DRIVE_ROOT) / 'output/data'

for name in ['Places.csv', 'rating_matrix_foody_users.csv', 'rating_matrix_foody_items.csv']:
    path = data_dir / name
    print('\n===', name, '===')
    if path.exists():
        df = pd.read_csv(path, nrows=5)
        print('path:', path)
        display(df)
    else:
        print('missing:', path)

## Chạy dry-run nếu chỉ muốn kiểm tra

Cell này chỉ để debug hoặc so sánh. Nó train và ghi Drive nhưng không upload R2.

In [ ]:
RUN_DRY_RUN_ONLY = False

if RUN_DRY_RUN_ONLY:
    !python colab_retrain_pipeline.py --force --dry-run
else:
    print('Bỏ qua dry-run. Cell train chính phía trên đã upload R2.')